# kin_01 — Population-level tongue movement statistics

What are the distributions of tongue movement parameters across all sessions?

**Pipeline:**
1. Load `all_tongue_movements` parquet
2. Quality filter (outbound phase fields required)
3. Distribution histograms for all kinematic parameters
4. Lick vs no-lick movement profiles
5. Session-level summary statistics
6. Pairwise kinematic scatter

## 1. Setup

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotstyle import apply_style, PALETTE, style_ax, save_fig
apply_style()

In [ ]:
if Path("/root/capsule").exists():
    ENV       = "codeocean"
    SCRATCH   = Path("/root/capsule/scratch")
    FOR_LOCAL = SCRATCH / "for_local"
else:
    ENV       = "local"
    FOR_LOCAL = Path("/Users/mib/Documents/Code/kinematics_analysis/data/for_local")
    SCRATCH   = FOR_LOCAL.parent

FIG_DIR  = SCRATCH / "figures" / "kin_01_population"
SAVE_FIG = False
print(f"ENV={ENV}")

## 2. Load data

In [ ]:
if ENV == "codeocean":
    movements_path = SCRATCH / "all_tongue_movements_04022026" / "all_tongue_movements_04022026.parquet"
else:
    movements_path = FOR_LOCAL / "all_tongue_movements_04022026.parquet"

all_tongue_movements = pd.read_parquet(movements_path)
print("Shape:", all_tongue_movements.shape)
print("Sessions:", all_tongue_movements["session"].nunique())

## 3. Quality filter

In [ ]:
# Require outbound phase fields and quality bounds
df = all_tongue_movements.dropna(subset=[
    "out_duration", "out_peak_velocity", "out_total_distance"
]).copy()

# Remove obvious outliers (>5 s duration, zero distance)
df = df[(df["out_duration"] > 0.05) & (df["out_duration"] < 5.0)]
df = df[(df["out_total_distance"] > 0) & (df["out_peak_velocity"] > 0)]

print(f"After filter: {len(df):,} / {len(all_tongue_movements):,} movements")
print(f"Sessions: {df['session'].nunique()}")

## 3b. Filter threshold sensitivity

How much data survives at different cutoff values for each quality-filter
criterion? Curves are marginal (one threshold varied at a time, holding the
rest of the data at the dropna-only stage) — they show sensitivity to each
cutoff in isolation, not the combined effect of all filters together.

In [ ]:
# Marginal retention curves for each filter threshold
base = all_tongue_movements.dropna(subset=[
    "out_duration", "out_peak_velocity", "out_total_distance"
]).copy()
n_base = len(base)

CURRENT_THRESH = {
    "out_duration_min":      0.05,
    "out_duration_max":      5.0,
    "out_peak_velocity_min": 0.0,
    "out_total_distance_min": 0.0,
}

def _retention_curve(ax, col, thresholds, mode, current, label):
    if mode == "min":
        pct = [100 * (base[col] >= t).mean() for t in thresholds]
    else:
        pct = [100 * (base[col] <= t).mean() for t in thresholds]
    ax.plot(thresholds, pct, color=PALETTE["neutral"])
    ax.axvline(current, color=PALETTE["neg"], ls="--", lw=1.2,
               label=f"current={current:g}")
    ax.set_xlabel(label)
    ax.set_ylabel("% retained")
    ax.legend(fontsize=7)
    style_ax(ax)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

dur = base["out_duration"]
_retention_curve(axes[0], "out_duration",
                  np.linspace(0, np.nanpercentile(dur, 25), 50),
                  "min", CURRENT_THRESH["out_duration_min"], "Duration min cutoff (s)")
_retention_curve(axes[1], "out_duration",
                  np.linspace(np.nanpercentile(dur, 50), np.nanpercentile(dur, 99.9), 50),
                  "max", CURRENT_THRESH["out_duration_max"], "Duration max cutoff (s)")

pv = base["out_peak_velocity"]
_retention_curve(axes[2], "out_peak_velocity",
                  np.linspace(0, np.nanpercentile(pv, 25), 50),
                  "min", CURRENT_THRESH["out_peak_velocity_min"], "Peak velocity min cutoff")

dist = base["out_total_distance"]
_retention_curve(axes[3], "out_total_distance",
                  np.linspace(0, np.nanpercentile(dist, 25), 50),
                  "min", CURRENT_THRESH["out_total_distance_min"], "Distance min cutoff")

fig.suptitle(f"Filter threshold sensitivity (n={n_base:,} movements with valid outbound fields)")
plt.tight_layout()
save_fig(fig, "filter_threshold_sensitivity", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 4. Kinematic distributions

In [ ]:
FEATS = {
    "out_peak_velocity":   "Peak velocity (a.u.)",
    "out_mean_velocity":   "Mean velocity (a.u.)",
    "out_duration":        "Out-phase duration (s)",
    "out_total_distance":  "Total distance (a.u.)",
    "excursion_angle_deg": "Excursion angle (°)",
    "endpoint_x":          "Endpoint X (px)",
    "endpoint_y":          "Endpoint Y (px)",
}

# use available columns
feat_items = [(k, v) for k, v in FEATS.items() if k in df.columns]
n_feats = len(feat_items)
cols = 3
rows = (n_feats + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = axes.flat

for ax, (col, label) in zip(axes, feat_items):
    vals = df[col].dropna().to_numpy()
    lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
    vals_c = vals[(vals >= lo) & (vals <= hi)]
    ax.hist(vals_c, bins=50, color=PALETTE["neutral"], edgecolor="white", linewidth=0.3)
    ax.axvline(np.median(vals_c), color=PALETTE["neg"], lw=1.2, ls="--",
               label=f"median={np.median(vals_c):.3g}")
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.legend(fontsize=7)
    style_ax(ax)

for ax in axes:
    ax.set_visible(False)
fig.suptitle("Movement parameter distributions (1–99th pctile, outbound-filtered)")
plt.tight_layout()
save_fig(fig, "movement_distributions", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 5. Lick vs no-lick profiles

In [ ]:
# Lick vs no-lick movement profiles
lick_col = "has_lick"
if lick_col in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for ax, col, label in [
        (axes[0], "out_peak_velocity", "Peak velocity"),
        (axes[1], "out_duration",      "Out-phase duration (s)"),
    ]:
        for lick_val, color, lbl in [
            (True,  PALETTE["pos"],  "Lick"),
            (False, PALETTE["neg"],  "No lick"),
        ]:
            vals = df[df[lick_col] == lick_val][col].dropna().to_numpy()
            if len(vals) == 0:
                continue
            lo, hi = np.nanpercentile(vals, 1), np.nanpercentile(vals, 99)
            vals_c = vals[(vals >= lo) & (vals <= hi)]
            ax.hist(vals_c, bins=40, color=color, alpha=0.6, label=lbl,
                    edgecolor="white", linewidth=0.3, density=True)
        ax.set_xlabel(label)
        ax.set_ylabel("Density")
        ax.legend()
        style_ax(ax)
    fig.suptitle("Lick vs no-lick movement profiles")
    plt.tight_layout()
    save_fig(fig, "lick_vs_nolick", fig_dir=FIG_DIR, save=SAVE_FIG)
    plt.show()

## 6. Session-level summaries

In [ ]:
# Session-level summary: movement counts and median kinematics
sess_summary = (
    df.groupby("session")
    .agg(
        n_movements=("out_duration", "count"),
        med_duration=("out_duration", "median"),
        med_peak_vel=("out_peak_velocity", "median"),
        med_distance=("out_total_distance", "median"),
    )
    .reset_index()
)
print(sess_summary.describe().round(2))

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (col, label) in zip(axes, [
    ("med_duration", "Median out-duration (s)"),
    ("med_peak_vel", "Median peak velocity"),
    ("med_distance", "Median total distance"),
]):
    ax.hist(sess_summary[col], bins=20, color=PALETTE["neutral"], edgecolor="white")
    ax.set_xlabel(label)
    ax.set_ylabel("Sessions")
    style_ax(ax)
fig.suptitle("Session-level kinematic medians")
plt.tight_layout()
save_fig(fig, "session_level_summaries", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()

## 7. Pairwise scatter

In [ ]:
# Pairwise scatter: do fast movements go farther?
SCATTER_PAIRS = [
    ("out_peak_velocity", "out_total_distance", "Peak vel", "Total distance"),
    ("out_duration",      "out_total_distance", "Duration (s)", "Total distance"),
    ("excursion_angle_deg","out_peak_velocity","Angle (°)", "Peak vel"),
]
SCATTER_PAIRS = [(a, b, la, lb) for a, b, la, lb in SCATTER_PAIRS
                 if a in df.columns and b in df.columns]

fig, axes = plt.subplots(1, len(SCATTER_PAIRS), figsize=(5 * len(SCATTER_PAIRS), 4))
if len(SCATTER_PAIRS) == 1:
    axes = [axes]
for ax, (cx, cy, lx, ly) in zip(axes, SCATTER_PAIRS):
    samp = df[[cx, cy]].dropna().sample(min(3000, len(df)), random_state=42)
    ax.scatter(samp[cx], samp[cy], s=2, alpha=0.2, color=PALETTE["neutral"], rasterized=True)
    ax.set_xlabel(lx)
    ax.set_ylabel(ly)
    style_ax(ax)
fig.suptitle("Kinematic pairwise scatter (3 k sample)")
plt.tight_layout()
save_fig(fig, "pairwise_scatter", fig_dir=FIG_DIR, save=SAVE_FIG)
plt.show()